In [3]:
# 06_inference_experiments.ipynb
# Purpose:
# - Experimentally run inference on the test set
# - Load latest champion model from registry
# - Keep everything in memory (no CSV saving)
# - Preview predictions

import pandas as pd
from pathlib import Path
import joblib
import logging

# ------------------------------
# Logging setup
# ------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ------------------------------
# Paths
# ------------------------------
BASE_DIR = Path.cwd().parent
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_REGISTRY_PATH = BASE_DIR / "models_artifacts"

TEST_CSV = PROCESSED_DATA_DIR / "test_features_final.csv"

# ------------------------------
# Load test dataset
# ------------------------------
test_df = pd.read_csv(TEST_CSV)
logger.info("Test dataset loaded: %s", test_df.shape)

# Keep DataFrame to preserve feature names
X_test = test_df

# ------------------------------
# Load latest champion model from registry
# ------------------------------
model_folders = sorted(
    [f for f in MODEL_REGISTRY_PATH.iterdir() if f.is_dir()],
    key=lambda x: x.name.split("_")[-1],  # assumes timestamp in folder name
    reverse=True
)

if not model_folders:
    raise FileNotFoundError("No models found in registry.")

latest_model_folder = model_folders[0]
model_path = latest_model_folder / "model.pkl"

model = joblib.load(model_path)
logger.info("Loaded latest model: %s", latest_model_folder.name)

# ------------------------------
# Generate predictions
# ------------------------------
if hasattr(model, "predict_proba"):
    y_pred = model.predict_proba(X_test)[:, 1]
else:
    y_pred = model.predict(X_test)

logger.info("Predictions generated. Sample:")
print(y_pred[:10])  # preview first 10 predictions


2026-02-08 15:34:29,405 | INFO | Test dataset loaded: (48744, 59)
2026-02-08 15:34:29,411 | INFO | Loaded latest model: logistic_regression_champion_20260125_135447
2026-02-08 15:34:29,421 | INFO | Predictions generated. Sample:


[0.32761255 0.63694871 0.34391271 0.34283069 0.68295362 0.35896076
 0.33306523 0.508405   0.22382814 0.67313333]
